# film revenue prediction - project notebook

## sections
1. project overview
2. data acquisition
3. exploratory data analysis
4. data merging
5. data cleaning
6. train / val / test split
7. data leakage analysis
8. feature engineering
9. multimodal fusion
10. baseline model
11. evaluation metrics
12. model selection + hyperparameter tuning
13. ablation study
14. fine-tuning experiment
15. interpretability with shap
16. report-ready outputs
17. pitfalls checklist

In [ ]:
# setup
import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 1) project overview

### 1.1 central research question
- what matters more for pre-release prediction: feature representation or model choice?

### 1.2 prediction target
- primary target: `y = log1p(revenue)`
- derived label: `profitable = 1[revenue > 1.5 * budget]`

### 1.3 core thesis
- combining structured metadata + synopsis embeddings + poster embeddings should outperform any single modality.

## 2) data acquisition

### 2.1 tmdb (kaggle)
- `TMDB_movie_dataset_v11.csv`

### 2.2 imdb non-commercial files
- `title.crew.tsv.csv` (gzipped)
- `title.principals.tsv.csv` (gzipped)
- `name.basics.tsv.csv` (gzipped)

### 2.3 poster images
- build url: `https://image.tmdb.org/t/p/w342/{poster_path}`
- cache locally before embedding extraction

In [ ]:
# load raw datasets
TMDB_PATH       = ROOT / "TMDB_movie_dataset_v11.csv"
CREW_PATH       = ROOT / "title.crew.tsv.csv"
PRINCIPALS_PATH = ROOT / "title.principals.tsv.csv"
NAMES_PATH      = ROOT / "name.basics.tsv.csv"

tmdb       = pd.read_csv(TMDB_PATH)
crew       = pd.read_csv(CREW_PATH,       sep='\t', compression='gzip', low_memory=False)
principals = pd.read_csv(PRINCIPALS_PATH, sep='\t', compression='gzip', low_memory=False)
names      = pd.read_csv(NAMES_PATH,      sep='\t', compression='gzip', low_memory=False)

print(f"tmdb:       {tmdb.shape}")
print(f"crew:       {crew.shape}")
print(f"principals: {principals.shape}")
print(f"names:      {names.shape}")
tmdb.head()

## 3) exploratory data analysis

- distributions of budget, revenue, and log-revenue
- release year trends
- genre frequency
- correlation heatmap of numeric features

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# NOTE: run this after section 5 (cleaning) since it uses the clean df
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["y_log_revenue"], bins=50, edgecolor='k')
axes[0].set_title("log1p(revenue) distribution")
axes[0].set_xlabel("log1p(revenue)")

axes[1].hist(np.log1p(df["budget"]), bins=50, edgecolor='k', color='orange')
axes[1].set_title("log1p(budget) distribution")
axes[1].set_xlabel("log1p(budget)")

df["release_year"] = df["release_date"].dt.year
axes[2].hist(df["release_year"], bins=40, edgecolor='k', color='green')
axes[2].set_title("release year distribution")
axes[2].set_xlabel("year")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_distributions.png", dpi=100)
plt.show()

# correlation heatmap — vote columns included here for EDA only, not used as features
numeric_cols = ["budget", "runtime", "popularity", "vote_count", "vote_average", "y_log_revenue"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("correlation heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_correlation.png", dpi=100)
plt.show()

# genre frequency
genre_counts = df["genres"].dropna().str.split(", ").explode().value_counts().head(15)
print("top 15 genres:")
print(genre_counts)

## 4) data merging

- use tmdb as base table
- join tmdb `imdb_id` to imdb `tconst`
- keep join diagnostics: row counts and match rate

> note: TMDB v11 has ~1.4M rows including TV/shorts — low overall match rate is expected.
> match rate on movies with valid budget+revenue (after section 5) will be much higher.

In [ ]:
## 3.1 merge tmdb + crew (directors & writers)
# tmdb is the base table; crew adds director/writer nconst lists per title
base = tmdb.copy()
n_base = len(base)

crew_small = crew[["tconst", "directors", "writers"]].copy()

# left join so all tmdb rows are kept; unmatched rows get NaN for crew columns
merged = base.merge(crew_small, how="left", left_on="imdb_id", right_on="tconst")
merged.drop(columns=["tconst"], inplace=True)  # redundant after join

crew_match = merged["directors"].notna().mean()
print(f"tmdb rows:        {n_base:,}")
print(f"after crew join:  {len(merged):,}")
print(f"crew match rate:  {crew_match:.2%}")  # low overall because ~53% of tmdb rows have no imdb_id

## 3.2 merge top-billed cast from principals
# principals is many-to-one (many cast members per title), so we aggregate first
# keep only actors/actresses ranked 1-3 to avoid exploding rows on join
cast = (
    principals[principals["category"].isin(["actor", "actress"])]
    .sort_values("ordering")
    .groupby("tconst")
    .head(3)
)

# pivot long → wide so each title gets one row with cast_1, cast_2, cast_3 nconst ids
cast["rank"] = cast.groupby("tconst").cumcount() + 1
cast_wide = (
    cast.pivot(index="tconst", columns="rank", values="nconst")
    .rename(columns={1: "cast_1", 2: "cast_2", 3: "cast_3"})
    .reset_index()
)

merged = merged.merge(cast_wide, how="left", left_on="imdb_id", right_on="tconst")
merged.drop(columns=["tconst"], inplace=True)

cast_match = merged["cast_1"].notna().mean()
print(f"after cast join:  {len(merged):,}")
print(f"cast match rate:  {cast_match:.2%}")  # ~81% among rows with imdb_id

## 3.3 resolve nconst → name for director and top cast
# nconst are opaque ids; map them to human-readable names for EDA and features
name_map = names.set_index("nconst")["primaryName"]

def resolve_names(nconst_series):
    return nconst_series.map(name_map)

# directors field can contain a comma-separated list; take only the first (primary) director
merged["director_name"] = resolve_names(merged["directors"].str.split(",").str[0])
merged["cast_1_name"]   = resolve_names(merged["cast_1"])
merged["cast_2_name"]   = resolve_names(merged["cast_2"])
merged["cast_3_name"]   = resolve_names(merged["cast_3"])

print(f"\nfinal shape: {merged.shape}")
merged[["title", "imdb_id", "director_name", "cast_1_name", "cast_2_name", "cast_3_name"]].head()

In [ ]:
# diagnose: how many tmdb rows actually have an imdb_id?
has_imdb = tmdb["imdb_id"].notna() & (tmdb["imdb_id"] != "")
print(f"tmdb rows with imdb_id:    {has_imdb.sum():,} / {len(tmdb):,} ({has_imdb.mean():.2%})")

# of those, how many matched crew?
has_imdb_merged = merged["imdb_id"].notna() & (merged["imdb_id"] != "")
crew_matched = merged["directors"].notna()
print(f"crew match rate (with id): {(crew_matched & has_imdb_merged).sum() / has_imdb_merged.sum():.2%}")

cast_matched = merged["cast_1"].notna()
print(f"cast match rate (with id): {(cast_matched & has_imdb_merged).sum() / has_imdb_merged.sum():.2%}")

## 5) data cleaning

- remove invalid/missing budget or revenue rows
- parse dates and normalize schema
- define train-ready base table

In [ ]:
df = merged.copy()
print(f"start:                    {len(df):,}")

# keep only released films — TMDB includes upcoming, in-production, and cancelled
# titles which have no revenue by definition and would just be noise
df = df[df["status"] == "Released"]
print(f"after status filter:      {len(df):,}")

# drop adult content — adult films follow different market dynamics and mixing
# them in would hurt the model's ability to learn patterns for mainstream movies
df = df[df["adult"] == False]
print(f"after adult filter:       {len(df):,}")

# require valid budget and revenue — TMDB uses 0 as a placeholder for "unknown",
# not an actual value, so 0 means missing data rather than a real zero
df = df[df["budget"].fillna(0) > 0]
df = df[df["revenue"].fillna(0) > 0]
print(f"after budget/rev > 0:     {len(df):,}")

# drop implausibly small values — entries like $1 budget or $500 revenue are
# clearly data entry errors; $10k is a conservative lower bound for any real
# theatrical release worth modeling
df = df[(df["budget"] >= 10_000) & (df["revenue"] >= 10_000)]
print(f"after min $10k filter:    {len(df):,}")

# parse and require a valid release date — release timing is used as a feature
# (season, year) and is also needed for the temporal cutoff in talent features
# (section 5), so rows without a date can't be used
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])
print(f"after date filter:        {len(df):,}")

# drop duplicates on tmdb id — some titles appear more than once in TMDB
# (e.g. different regional entries); keep first to avoid inflating the training set
df = df.drop_duplicates(subset=["id"])
print(f"after dedup:              {len(df):,}")

# require a non-empty overview — synopsis embeddings are a planned feature group
# (section 5); rows with no overview can't contribute to that modality so we
# drop them now rather than impute
df = df[df["overview"].fillna("").str.strip() != ""]
print(f"after overview filter:    {len(df):,}")

# define targets
df["y_log_revenue"] = np.log1p(df["revenue"])
df["profitable"]    = (df["revenue"] > 1.5 * df["budget"]).astype(int)

print(f"\nfinal clean shape: {df.shape}")
print(f"profitable share:  {df['profitable'].mean():.2%}")
df[["title", "release_date", "budget", "revenue", "y_log_revenue", "profitable"]].head()

## 6) train / val / test split

- split before any feature fitting to prevent data leakage from test set into scalers/encoders
- temporal split: train on older films, test on newer — more realistic than random split
- 70% train / 15% val / 15% test

In [ ]:
# sort by release date so older films train the model and newer films test it
df_sorted = df.sort_values("release_date").reset_index(drop=True)

n         = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df_sorted.iloc[:train_end].copy()
val_df   = df_sorted.iloc[train_end:val_end].copy()
test_df  = df_sorted.iloc[val_end:].copy()

print(f"train: {len(train_df):,}  ({train_df['release_date'].min().year}–{train_df['release_date'].max().year})")
print(f"val:   {len(val_df):,}  ({val_df['release_date'].min().year}–{val_df['release_date'].max().year})")
print(f"test:  {len(test_df):,}  ({test_df['release_date'].min().year}–{test_df['release_date'].max().year})")

## 7) data leakage analysis

- define the pre-release information boundary
- columns like `vote_average`, `vote_count`, `popularity` reflect post-release audience response — exclude from main features
- run a leakage demonstration: train with vs without these columns to quantify their inflation effect

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# columns known before a film's release date — safe to use as features
PRE_RELEASE_COLS = [
    "budget", "runtime", "original_language",
    "genres", "production_companies", "production_countries",
    "release_date",   # used to derive timing features
    "overview",       # for synopsis embeddings
    "poster_path",    # for poster embeddings
    "director_name", "cast_1_name", "cast_2_name", "cast_3_name",
    "directors", "cast_1", "cast_2", "cast_3",
]

# only known after release — must not be used as predictive features
POST_RELEASE_COLS = ["vote_average", "vote_count", "popularity"]

print("post-release columns excluded from main feature set:")
for col in POST_RELEASE_COLS:
    print(f"  {col}")

# leakage demonstration: compare r2 with and without post-release columns
demo_safe   = ["budget", "runtime"]
demo_leaked = ["budget", "runtime", "vote_count", "vote_average", "popularity"]

def demo_r2(cols):
    _train = train_df[cols + ["y_log_revenue"]].dropna()
    _val   = val_df[cols + ["y_log_revenue"]].dropna()
    model  = Ridge().fit(_train[cols], _train["y_log_revenue"])
    return r2_score(_val["y_log_revenue"], model.predict(_val[cols]))

print(f"\nr2 without leakage cols: {demo_r2(demo_safe):.3f}")
print(f"r2 WITH leakage cols:    {demo_r2(demo_leaked):.3f}")
print("→ inflated r2 confirms these are post-release signals, not predictive features")

## 8) feature engineering

### 8.1 structured metadata
- log-budget, runtime, release month/year, language indicator

### 8.2 genre indicators
- one-hot encode comma-separated genres (handcrafted)

### 8.3 synopsis embeddings
- encode `overview` text with a pretrained sentence transformer (all-MiniLM-L6-v2, 384-dim)
- cache to disk — only computed once

### 8.4 poster embeddings
- download poster images from TMDB and encode with ResNet18 (512-dim)
- cache to disk — slow on first run (~10–30 min for 10k images)

In [ ]:
## 8.1 structured metadata features
# fit scaler on train only — applying to val/test prevents leakage from their distributions

for split in [train_df, val_df, test_df]:
    split["log_budget"]    = np.log1p(split["budget"])
    split["release_month"] = split["release_date"].dt.month
    split["release_year"]  = split["release_date"].dt.year
    split["is_english"]    = (split["original_language"] == "en").astype(int)

structured_cols = ["log_budget", "runtime", "release_month", "release_year", "is_english"]

scaler_struct  = StandardScaler()
X_struct_train = scaler_struct.fit_transform(train_df[structured_cols].fillna(0))
X_struct_val   = scaler_struct.transform(val_df[structured_cols].fillna(0))
X_struct_test  = scaler_struct.transform(test_df[structured_cols].fillna(0))

print(f"8.1 structured features: {X_struct_train.shape}")

In [ ]:
## 8.2 genre indicators
# one-hot encode the comma-separated genres string into binary columns
# fit column list on train only — val/test unseen genres are dropped

def genre_dummies(split_df):
    return split_df["genres"].fillna("").str.get_dummies(sep=", ")

g_train = genre_dummies(train_df)
g_val   = genre_dummies(val_df)
g_test  = genre_dummies(test_df)

genre_cols    = g_train.columns.tolist()  # only genres seen in training
X_genre_train = g_train.values
X_genre_val   = g_val.reindex(columns=genre_cols, fill_value=0).values
X_genre_test  = g_test.reindex(columns=genre_cols, fill_value=0).values

print(f"8.2 genre features: {X_genre_train.shape}  —  {genre_cols}")

In [ ]:
## 8.3 synopsis embeddings
# encode movie overviews with a pretrained sentence transformer
# all-MiniLM-L6-v2 is fast and produces 384-dim vectors — good balance of quality vs speed
# embeddings are cached to disk so this cell only runs the model once

from sentence_transformers import SentenceTransformer

SYNOPSIS_EMB_PATH = OUTPUT_DIR / "synopsis_embeddings.npy"
SYNOPSIS_ID_PATH  = OUTPUT_DIR / "synopsis_ids.npy"

if SYNOPSIS_EMB_PATH.exists():
    synopsis_embs = np.load(SYNOPSIS_EMB_PATH)
    synopsis_ids  = np.load(SYNOPSIS_ID_PATH)
    print(f"loaded cached synopsis embeddings: {synopsis_embs.shape}")
else:
    st_model      = SentenceTransformer("all-MiniLM-L6-v2")
    overviews     = df_sorted["overview"].fillna("").tolist()
    synopsis_embs = st_model.encode(overviews, batch_size=64, show_progress_bar=True)
    synopsis_ids  = df_sorted["id"].values
    np.save(SYNOPSIS_EMB_PATH, synopsis_embs)
    np.save(SYNOPSIS_ID_PATH, synopsis_ids)
    print(f"computed and cached synopsis embeddings: {synopsis_embs.shape}")

# align embeddings back to each split by movie id
def align_emb(split_df, all_ids, all_embs):
    id_to_emb = dict(zip(all_ids, all_embs))
    dim = all_embs.shape[1]
    return np.array([id_to_emb.get(i, np.zeros(dim)) for i in split_df["id"]])

X_synopsis_train = align_emb(train_df, synopsis_ids, synopsis_embs)
X_synopsis_val   = align_emb(val_df,   synopsis_ids, synopsis_embs)
X_synopsis_test  = align_emb(test_df,  synopsis_ids, synopsis_embs)

print(f"8.3 synopsis embeddings: {X_synopsis_train.shape}")

In [ ]:
## 8.4 poster embeddings
# download poster images from TMDB and encode with ResNet18 (classification head removed → 512-dim)
# slow on first run (~10–30 min for 10k images) — cached to disk afterward

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

POSTER_EMB_PATH = OUTPUT_DIR / "poster_embeddings.npy"
POSTER_ID_PATH  = OUTPUT_DIR / "poster_ids.npy"

if POSTER_EMB_PATH.exists():
    poster_embs = np.load(POSTER_EMB_PATH)
    poster_ids  = np.load(POSTER_ID_PATH)
    print(f"loaded cached poster embeddings: {poster_embs.shape}")
else:
    resnet = models.resnet18(weights="IMAGENET1K_V1")
    resnet.fc = torch.nn.Identity()  # remove final layer to get 512-dim features
    resnet.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    poster_vecs, valid_ids = [], []
    for _, row in df_sorted.iterrows():
        if pd.isna(row["poster_path"]):
            continue
        url = f"https://image.tmdb.org/t/p/w342{row['poster_path']}"
        try:
            img = Image.open(BytesIO(requests.get(url, timeout=5).content)).convert("RGB")
            with torch.no_grad():
                vec = resnet(transform(img).unsqueeze(0)).squeeze().numpy()
            poster_vecs.append(vec)
            valid_ids.append(row["id"])
        except Exception:
            continue  # skip missing/broken images gracefully

    poster_embs = np.array(poster_vecs)
    poster_ids  = np.array(valid_ids)
    np.save(POSTER_EMB_PATH, poster_embs)
    np.save(POSTER_ID_PATH, poster_ids)
    print(f"computed and cached poster embeddings: {poster_embs.shape}")

X_poster_train = align_emb(train_df, poster_ids, poster_embs)
X_poster_val   = align_emb(val_df,   poster_ids, poster_embs)
X_poster_test  = align_emb(test_df,  poster_ids, poster_embs)

print(f"8.4 poster embeddings: {X_poster_train.shape}")

## 9) multimodal fusion

- early fusion: concatenate all modality matrices into one design matrix
- this lets any downstream model learn cross-modal interactions
- individual modality matrices are kept separate for the ablation study (section 13)

In [ ]:
# early fusion: horizontally stack all modality feature matrices
# order: structured | genre | synopsis | poster
X_train_full = np.hstack([X_struct_train, X_genre_train, X_synopsis_train, X_poster_train])
X_val_full   = np.hstack([X_struct_val,   X_genre_val,   X_synopsis_val,   X_poster_val])
X_test_full  = np.hstack([X_struct_test,  X_genre_test,  X_synopsis_test,  X_poster_test])

y_train = train_df["y_log_revenue"].values
y_val   = val_df["y_log_revenue"].values
y_test  = test_df["y_log_revenue"].values

print(f"full fused feature matrix: {X_train_full.shape}")
print(f"  structured: {X_struct_train.shape[1]} dims")
print(f"  genre:      {X_genre_train.shape[1]} dims")
print(f"  synopsis:   {X_synopsis_train.shape[1]} dims")
print(f"  poster:     {X_poster_train.shape[1]} dims")

## 10) baseline model

- ridge regression on structured metadata only (no embeddings, no talent features)
- establishes the performance floor that more complex models must beat

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

baseline_cols = ["budget", "runtime"]

train_base = train_df[baseline_cols + ["y_log_revenue"]].dropna()
val_base   = val_df[baseline_cols + ["y_log_revenue"]].dropna()

scaler = StandardScaler()
X_train = scaler.fit_transform(train_base[baseline_cols])
X_val   = scaler.transform(val_base[baseline_cols])

baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_train, train_base["y_log_revenue"])

y_pred = baseline_model.predict(X_val)
y_true = val_base["y_log_revenue"]

print("baseline — ridge on budget + runtime:")
print(f"  rmse: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"  mae:  {mean_absolute_error(y_true, y_pred):.4f}")
print(f"  r2:   {r2_score(y_true, y_pred):.4f}")

## 11) evaluation metrics

- primary regression metrics: rmse, mae, r2 on log-revenue
- secondary classification metrics: precision, recall, f1, auc on profitability label
- all metrics computed on val set during development; test set used only for final evaluation

In [ ]:
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

def evaluate_regression(y_true, y_pred, label="model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{label} — rmse: {rmse:.4f} | mae: {mae:.4f} | r2: {r2:.4f}")
    return {"rmse": rmse, "mae": mae, "r2": r2}

def evaluate_classification(y_true_binary, y_pred_proba, label="model"):
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    p   = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    r   = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1  = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    auc = roc_auc_score(y_true_binary, y_pred_proba)
    print(f"{label} — precision: {p:.3f} | recall: {r:.3f} | f1: {f1:.3f} | auc: {auc:.3f}")
    return {"precision": p, "recall": r, "f1": f1, "auc": auc}

print("evaluate_regression() and evaluate_classification() defined — call per experiment")

## 12) model selection + hyperparameter tuning

- tree ensembles: random forest, xgboost, lightgbm
- tune best candidate with cross-validation on val set

## 13) ablation study

- fixed temporal split and evaluation protocol
- experiment matrix comparing modality contributions:
  - e0: structured only (baseline)
  - e1: + genre indicators
  - e2: + synopsis embeddings
  - e3: + poster embeddings
  - e4: all modalities combined

## 14) fine-tuning experiment

- fine-tune one transfer model variant and compare against frozen baseline

## 15) interpretability with shap

- explain best model globally and locally

## 16) report-ready outputs

- export key figures and tables for final report

## 17) pitfalls checklist

- no leakage: all post-release signals excluded from training features
- temporal consistency: talent history features use strict pre-release cutoff
- reproducible splits and seeds: SEED=42 throughout
- test set locked: used only for final evaluation, never for model selection